In [1]:
import rasterio
import rioxarray
import xarray as xr
import pandas as pd
from rasterio.merge import merge
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterio.features import rasterize
from sklearn.metrics import confusion_matrix

In [2]:
def build_union_grid(rasters, res=None):
    """Compute union extent and create a target grid."""
    bounds = []
    resolutions = []
    for f in rasters:
        da = rioxarray.open_rasterio(f).squeeze()
        bounds.append(da.rio.bounds())
        resolutions.append(da.rio.resolution())
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)

    # Pick resolution (finest among all rasters if not specified)
    if res is None:
        resx = min(abs(r[0]) for r in resolutions)
        resy = min(abs(r[1]) for r in resolutions)
    else:
        resx, resy = res

    # Build coordinates
    xs = np.arange(minx, maxx + resx, resx)
    ys = np.arange(maxy, miny - resy, -resy)  # descending
    target = xr.DataArray(
        np.empty((len(ys), len(xs))),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    target.rio.write_crs("EPSG:4326", inplace=True)
    return target

def merge_rasters_with_priority_nonzero(rasters, priority_order=None, res=None, nodata_value=None):
    """
    Merge multiple rasters onto a union grid with explicit priority,
    including only values that are not NoData and not zero.

    Parameters
    ----------
    rasters : list of str
        Paths to input rasters.
    priority_order : list of int, optional
        Indexes of rasters in `rasters` defining priority (0 = highest).
        If None, order in list is used.
    res : tuple of float, optional
        (xres, yres). If None, pick finest resolution among rasters.
    nodata_value : float, optional
        The nodata value of your rasters. If None, defaults to NaN.
    """
    if priority_order is None:
        priority_order = list(range(len(rasters)))

    # Step 1: Build union grid
    union_grid = build_union_grid(rasters, res=res)

    # Step 2: Reproject / resample each raster onto union grid
    arrays = []
    for idx in priority_order:
        da = rioxarray.open_rasterio(rasters[idx]).squeeze()
        if nodata_value is not None:
            da = da.where(da != nodata_value, np.nan)
        da_aligned = da.rio.reproject_match(union_grid)
        arrays.append(da_aligned)

    # Step 3: Apply priority, but only where data is nonzero and not NaN
    final = xr.full_like(union_grid, np.nan)
    for arr in arrays:
        mask = (~np.isnan(arr)) & (arr != 0) & np.isnan(final)  # only fill empty cells
        final = xr.where(mask, arr, final)

    return final

In [14]:
# Paths to your three rasters
rasters = ["D:/MyDrive/Stability/RawData/Oliver_direct/Strain_Files_2025/app_strain_S1AA_20170408T025731_20170502T025732_VVP024_INT80_G_ueF_29C7_wrapped_phase.tif",
           "D:/MyDrive/Stability/RawData/Oliver_direct/Strain_Files_2025/app_strain_S1BB_20170417T032145_20170429T032146_VVP012_INT80_G_ueF_78A9_wrapped_phase.tif",
           "D:/MyDrive/Stability/RawData/Oliver_direct/Strain_Files_2025/app_strain_S1BB_20170415T033809_20170509T033811_VVP024_INT80_G_ueF_98DA_wrapped_phase.tif"
                ]

merged = merge_rasters_with_priority_nonzero(rasters, nodata_value=0) 

# Save result
#merged.rio.to_raster("D:/MyDrive/Stability/RawData/Oliver_direct/Strain_Files_2025/merged_oliver.tif")

In [15]:
shapefile_path = "D:/MyDrive/Ice_stability_zones/merged_Oliver_zones.shp"

# Load the shapefile
gdf = gpd.read_file(shapefile_path)

In [16]:
# Define mapping of class names to integers
class_map = {"Bottomfast": 1, "Stabilized": 2, "Not Stabilzied": 3}
gdf['class_id'] = gdf['source'].map(class_map)

# Rasterize the shapefile to match the raster grid
truth_raster = rasterize(
    [(geom, value) for geom, value in zip(gdf.geometry, gdf['class_id'])],
    out_shape=merged.shape,
    transform=merged.rio.transform(),
    fill=-1,  # pixels outside polygons
    dtype=np.int32
)

In [17]:
lowerthreshold = 8.6e-06
upperthreshold = 2.4e-05
pred_raster = np.full_like(merged.values, -1, dtype=np.int32)

# Example: define bins
pred_raster[(merged.values > 0.000000000001) & (merged.values <= lowerthreshold)] = 1  # bottomfast
pred_raster[(merged.values > lowerthreshold) & (merged.values <= upperthreshold)] = 2  # stabilized
pred_raster[merged.values > upperthreshold] = 3   #not stabilized
mask = truth_raster != -1
y_true = truth_raster[mask]
y_pred = pred_raster[mask]

In [18]:
classes = [1, 2, 3]  # bottomfast, stabilized, not_stabilized
cm = confusion_matrix(y_true, y_pred, labels=classes)

cm_df = pd.DataFrame(cm, index=class_map.keys(), columns=class_map.keys())
cm_row_percent = cm_df.div(cm_df.sum(axis=1), axis=0) * 100
print(cm_row_percent)

                Bottomfast  Stabilized  Not Stabilzied
Bottomfast       41.795612   33.399965       24.804422
Stabilized       28.111276   46.556623       25.332101
Not Stabilzied    3.592561   22.792526       73.614914
